<a href="https://colab.research.google.com/github/vlaks524/DSCC-251-Final-Project/blob/main/DSCC_251_Final_Project_(RoBERTa%2C_ver1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.7 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report

import torch
from datasets import Dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [7]:
#Loading FinancialPhraseBank file (75 Agree)
FILE_PATH = "/content/Sentences_75Agree.txt"

# read raw lines
with open(FILE_PATH, "r", encoding="utf-8", errors="replace") as f:
    lines = f.readlines()

# parse "sentence@label"
rows = []
for line in lines:
    line = line.strip()
    if not line:
        continue

    # split from the right just in case "@" appears in text
    parts = line.rsplit("@", 1)
    if len(parts) != 2:
        continue

    text, label = parts
    text = text.strip()
    label = label.strip().lower()

    if text and label in ["positive", "neutral", "negative"]:
        rows.append((text, label))

df = pd.DataFrame(rows, columns=["text", "label"])

print("Shape:", df.shape)
print(df.head())
print("\nLabel counts:")
print(df["label"].value_counts())

Shape: (3453, 2)
                                                text     label
0  According to Gran , the company has no plans t...   neutral
1  With the new production plant the company woul...  positive
2  For the last quarter of 2010 , Componenta 's n...  positive
3  In the third quarter of 2010 , net sales incre...  positive
4  Operating profit rose to EUR 13.1 mn from EUR ...  positive

Label counts:
label
neutral     2146
positive     887
negative     420
Name: count, dtype: int64


In [8]:
#Encoding the labels
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

print("Classes:", list(label_encoder.classes_))
print(df.head())

Classes: ['negative', 'neutral', 'positive']
                                                text     label  label_id
0  According to Gran , the company has no plans t...   neutral         1
1  With the new production plant the company woul...  positive         2
2  For the last quarter of 2010 , Componenta 's n...  positive         2
3  In the third quarter of 2010 , net sales incre...  positive         2
4  Operating profit rose to EUR 13.1 mn from EUR ...  positive         2


In [9]:
#Splitting into test set and pool set
X = df["text"].tolist()
y = df["label_id"].tolist()

# fixed held-out test set
X_pool, X_test, y_pool, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Pool size:", len(X_pool))
print("Test size:", len(X_test))

Pool size: 2762
Test size: 691


In [10]:
#Creating initial labeled set and unlabeled pool
INITIAL_LABEL_SIZE = 60

pool_indices = np.arange(len(X_pool))

initial_indices, unlabeled_indices = train_test_split(
    pool_indices,
    train_size=INITIAL_LABEL_SIZE,
    random_state=42,
    stratify=np.array(y_pool)
)

X_labeled = [X_pool[i] for i in initial_indices]
y_labeled = [y_pool[i] for i in initial_indices]

X_unlabeled = [X_pool[i] for i in unlabeled_indices]
y_unlabeled = [y_pool[i] for i in unlabeled_indices]  # hidden during AL, but kept for oracle simulation

print("Initial labeled set:", len(X_labeled))
print("Unlabeled pool:", len(X_unlabeled))
print("Test set:", len(X_test))

Initial labeled set: 60
Unlabeled pool: 2702
Test set: 691


In [11]:
#RoBERTa tokenizer
MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
train_df = pd.DataFrame({
    "text": X_labeled,
    "label": y_labeled
})

test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test
})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
test_dataset.set_format("torch")

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Map:   0%|          | 0/691 [00:00<?, ? examples/s]

In [13]:
#Defining Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    f1_macro = f1_score(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1_macro": f1_macro
    }

In [14]:
#Building the model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_encoder.classes_)
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
#Training setup
training_args = TrainingArguments(
    output_dir="./roberta_phrasebank_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none"
)

In [18]:
#Training initial model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.055113,1.027846,0.620839,0.255357
2,0.969991,0.977073,0.620839,0.255357
3,0.918461,0.950568,0.620839,0.255357


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=24, training_loss=0.9811882972717285, metrics={'train_runtime': 1420.4649, 'train_samples_per_second': 0.127, 'train_steps_per_second': 0.017, 'total_flos': 11840103797760.0, 'train_loss': 0.9811882972717285, 'epoch': 3.0})

In [19]:
eval_results = trainer.evaluate()
print(eval_results)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 1.0273587703704834, 'eval_accuracy': 0.6208393632416788, 'eval_f1_macro': 0.2553571428571429, 'eval_runtime': 349.1654, 'eval_samples_per_second': 1.979, 'eval_steps_per_second': 0.126, 'epoch': 3.0}


In [20]:
pred_output = trainer.predict(test_dataset)
preds = np.argmax(pred_output.predictions, axis=1)

print("Macro F1:", f1_score(y_test, preds, average="macro"))
print("Accuracy:", accuracy_score(y_test, preds))
print("\nClassification report:\n")
print(classification_report(y_test, preds, target_names=label_encoder.classes_))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Macro F1: 0.2553571428571429
Accuracy: 0.6208393632416788

Classification report:

              precision    recall  f1-score   support

    negative       0.00      0.00      0.00        84
     neutral       0.62      1.00      0.77       429
    positive       0.00      0.00      0.00       178

    accuracy                           0.62       691
   macro avg       0.21      0.33      0.26       691
weighted avg       0.39      0.62      0.48       691



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
